# Inference demo

Zero-shot BLIP on a demo tile, then the same tile through the running laptop backend (`uvicorn app.backend.main:app`).

In [ ]:
import json, httpx
from PIL import Image
from farmvlm.blip import BlipReader
tiles = json.load(open('../data/demo_tiles/tiles.json'))
tile = tiles[0]; img = Image.open(f"../data/demo_tiles/{tile['id']}.jpg").convert('RGB').resize((384, 384))
reader = BlipReader(device='cpu')
print('truth:', tile['truth'])
print('BLIP caption:', reader.caption([img]))
print('BLIP answer:', reader.answer([img], ['Is there drydown in this tile?']))

In [ ]:
c = httpx.Client(base_url='http://127.0.0.1:8000', timeout=120)
with c.stream('POST', '/api/read', json={'image': {'tile_id': tile['id']}, 'model': 'ft'}) as r:
    done = [json.loads(l) for l in r.iter_lines()][-1]
print(done['caption']); print(done['findings'])